# Nemotron-3-Nano-30B — GRPO Polish on 85% SFT (v11)

Sophisticated GRPO post-training, designed for a strong (≈85%) SFT
starting point on a single RTX 6000 Pro 96 GB.

**Pipeline**
1. Load base + attach LoRA matching SFT config; warm-start from SFT-85 adapter.
2. (Optional) Micro-SFT on 4 k Sonnet 4.6 thinking traces (style graft).
3. Curriculum filter: keep prompts where SFT pass@1 ∈ [0.25, 0.85].
4. GRPO with vLLM colocate rollouts, DAPO asymmetric clip, refined reward.
5. Save + zip submission.

**Key design choices for an 85% starting point**
- LR **2e-6** (vs v9-1's 5e-6): high baseline → easy to overshoot.
- KL **β = 0.05** (vs 0.02): stronger anchor; prevents drift.
- `num_generations=8`, `max_completion_length=3072`: lower variance, full CoT room.
- DAPO asymmetric clip `epsilon_high=0.28`: rewards rare correct tokens.
- LoRA: rank 32 / α 64 / RSLoRA — **matches SFT exactly** (do not break what works).
- Drop Mamba `x_proj`/`dt_proj` from targets: SSM near-saturation, perturbation risk.
- Reward de-saturated: correctness 1.5 / format 0.3 / reasoning-quality 0.5 / length 0.2.
- Reference policy: `model.disable_adapter()` toggle — saves ~60 GB vs second model.


## Mode Selection

In [ ]:
import os, sys
os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ["TORCHINDUCTOR_DISABLE"] = "1"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="strict")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="strict")

# ---- Run modes -----------------------------------------------------------
TRAIN_ON_KAGGLE = 1   # 1 = train (Kaggle or local); 0 = package-only
USE_PRETRAINED  = 0
assert (TRAIN_ON_KAGGLE + USE_PRETRAINED) == 1

# ---- Stage toggles -------------------------------------------------------
DO_MICRO_SFT_SONNET = 1   # Pre-GRPO 1-epoch SFT on Sonnet 4.6 traces
DO_CURRICULUM_FILTER = 1  # Pre-score with SFT model; keep mid-difficulty
DO_GRPO              = 1  # The main event

# ---- Paths ---------------------------------------------------------------
# SFT-85 checkpoint to warm-start from (must contain adapter_config.json +
# adapter_model.safetensors).
SFT85_ADAPTER_PATH = os.environ.get(
    "SFT85_ADAPTER_PATH",
    "/kaggle/input/datasets/your-username/nemotron-sft-85",  # ← edit
)

# 4k Sonnet-4.6 thinking-trace SFT dataset (CSV with prompt + answer + cot)
SONNET_TRACES_PATH = os.environ.get(
    "SONNET_TRACES_PATH",
    "/kaggle/input/datasets/your-username/sonnet46-thinking-4k.csv",  # ← edit
)

# Main GRPO prompt set (only prompt + answer are used; model generates own CoT)
GRPO_DATASET_PATH = os.environ.get(
    "GRPO_DATASET_PATH",
    "/kaggle/input/datasets/dgxchen/nemotron-cot-tong/problem_ids_matched.csv",
)

BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
PRETRAINED_ADAPTER_DATASET_PATH = SFT85_ADAPTER_PATH  # used in package-only mode

print({
    "TRAIN_ON_KAGGLE": TRAIN_ON_KAGGLE,
    "DO_MICRO_SFT_SONNET": DO_MICRO_SFT_SONNET,
    "DO_CURRICULUM_FILTER": DO_CURRICULUM_FILTER,
    "DO_GRPO": DO_GRPO,
    "SFT85_ADAPTER_PATH": SFT85_ADAPTER_PATH,
    "SONNET_TRACES_PATH": SONNET_TRACES_PATH,
    "GRPO_DATASET_PATH": GRPO_DATASET_PATH,
})


## Setup & Model Loading

In [ ]:
import os, glob, sys, subprocess, site

candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
print("Found Triton wheels:", candidates)
if not candidates:
    raise FileNotFoundError("No Triton wheel found under /kaggle/input")
wheel = candidates[0]

target = "/kaggle/working/pydeps"
os.makedirs(target, exist_ok=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-deps", "--target", target,
     "--upgrade", "--ignore-installed", wheel],
    check=True,
)
if target not in sys.path:
    sys.path.insert(0, target)
site.addsitedir(target)
import importlib.util
print("triton spec:", importlib.util.find_spec("triton"))


In [ ]:
if TRAIN_ON_KAGGLE:
    import sys, os, shutil, stat
    sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')
    ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
    ptxas_dst = '/tmp/ptxas-blackwell'
    if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
        src_bin = os.path.dirname(ptxas_src)
        dst_bin = '/tmp/triton_nvidia_bin'
        shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
        for f in os.listdir(dst_bin):
            fp = os.path.join(dst_bin, f)
            if os.path.isfile(fp):
                os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
        os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst
        import triton.backends.nvidia as nv_backend
        nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
        os.environ['TRITON_PTXAS_PATH'] = ptxas_dst
    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: '12.0'
    print('Training environment fixes applied.')


In [ ]:
if TRAIN_ON_KAGGLE:
    import glob, os, subprocess, sys
    def recursive_wheels(pattern):
        return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    packages_dir = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"
    import torch
    print("Torch:", torch.__version__, " CUDA:", torch.version.cuda)
    if not torch.cuda.is_available():
        raise RuntimeError("GPU required.")
    if not os.path.isdir(packages_dir):
        raise FileNotFoundError(packages_dir)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "--no-index", "--find-links", packages_dir,
         "unsloth", "trl", "peft", "transformers", "datasets",
         "accelerate", "bitsandbytes"],
        check=True,
    )
    # Best-effort vLLM install — required for fast GRPO rollouts.
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q",
             "--no-index", "--find-links", packages_dir, "vllm"],
            check=True,
        )
        import vllm  # noqa: F401
        print("vLLM available — will use colocate rollouts.")
        VLLM_OK = True
    except Exception as e:
        print(f"vLLM unavailable, will fall back to HF generate: {e}")
        VLLM_OK = False
    all_mamba = recursive_wheels("mamba_ssm-*.whl")
    all_causal = recursive_wheels("causal*conv1d*.whl")
    if all_causal:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", all_causal[-1]], check=True)
    if all_mamba:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", all_mamba[-1]], check=True)
    else:
        raise FileNotFoundError("Missing mamba_ssm wheel.")
    print("Offline package installation finished.")


In [ ]:
if TRAIN_ON_KAGGLE:
    import torch, kagglehub
    from unsloth import FastLanguageModel
    try:
        from unsloth import PatchFastRL
        PatchFastRL("GRPO", FastLanguageModel)
        print("PatchFastRL applied for GRPO.")
    except (ImportError, AttributeError):
        print("PatchFastRL not available; proceeding without it.")

    MAX_SEQ_LEN = 8192
    MODEL_PATH = kagglehub.model_download(
        "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
    )
    print("Model path:", MODEL_PATH)

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_PATH,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=False,
        load_in_8bit=False,
        full_finetuning=False,
        trust_remote_code=True,
        unsloth_force_compile=False,
        attn_implementation="eager",
        dtype=torch.bfloat16,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print("Base model loaded.")


## LoRA Wrap + SFT-85 Warm-Start

In [ ]:
if TRAIN_ON_KAGGLE:
    from unsloth import FastLanguageModel
    from safetensors.torch import load_file
    import os, json

    # ------------------------------------------------------------------
    # LoRA config — MUST mirror the SFT-85 adapter we are warm-starting
    # from. Changing rank/alpha/targets here would silently break the
    # weight reload.
    # ------------------------------------------------------------------
    LORA_RANK    = 32      # contest cap
    LORA_ALPHA   = 64      # 2× rank; RSLoRA divides by sqrt(r) internally
    LORA_DROPOUT = 0.0

    # NOTE on target_modules:
    #   Keep attention + MoE/FFN; DROP Mamba x_proj / dt_proj for the GRPO
    #   polish stage — SSM is the most sensitive to small perturbations
    #   and at 85% it is already well-tuned by SFT. Polishing it with RL
    #   tends to destabilize generation.
    # If your SFT-85 adapter *did* include x_proj/dt_proj, set
    # INCLUDE_MAMBA_LORA=True so the warm-start matches its layout.
    INCLUDE_MAMBA_LORA = True   # safest: match SFT layout exactly

    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "in_proj", "out_proj",
        "gate_proj", "up_proj", "down_proj",
    ]
    if INCLUDE_MAMBA_LORA:
        target_modules += ["x_proj", "dt_proj"]

    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=target_modules,
        bias="none",
        use_gradient_checkpointing=True,
        random_state=42,
        use_rslora=True,
    )
    model.print_trainable_parameters()

    # ------------------------------------------------------------------
    # Warm-start from SFT-85 adapter
    # ------------------------------------------------------------------
    adapter_file = os.path.join(SFT85_ADAPTER_PATH, "adapter_model.safetensors")
    cfg_file     = os.path.join(SFT85_ADAPTER_PATH, "adapter_config.json")
    if not os.path.exists(adapter_file):
        raise FileNotFoundError(f"SFT-85 adapter not found: {adapter_file}")

    # Sanity-check the SFT adapter config vs our current LoRA wrap.
    with open(cfg_file) as f:
        sft_cfg = json.load(f)
    print("SFT-85 adapter_config preview:",
          {k: sft_cfg.get(k) for k in (
              "r", "lora_alpha", "use_rslora", "lora_dropout", "target_modules"
          )})
    if int(sft_cfg.get("r", -1)) != LORA_RANK or int(sft_cfg.get("lora_alpha", -1)) != LORA_ALPHA:
        print("WARNING: SFT adapter rank/alpha differs — reload may be partial.")

    sd = load_file(adapter_file)
    model_sd = model.state_dict()
    renamed = {}
    for k, v in sd.items():
        if k in model_sd:
            renamed[k] = v
        else:
            # PEFT typically stores under ".default" subkey; safetensors
            # files sometimes omit it.
            for adapt in ("lora_A", "lora_B"):
                needle = f"{adapt}.weight"
                if needle in k:
                    cand = k.replace(needle, f"{adapt}.default.weight")
                    if cand in model_sd:
                        renamed[cand] = v
                        break
    missing, unexpected = model.load_state_dict(renamed, strict=False)
    lora_missing = [m for m in missing if "lora_" in m]
    print(f"Warm-start: matched {len(renamed)} tensors, "
          f"LoRA-missing={len(lora_missing)}, LoRA-unexpected={len([u for u in unexpected if 'lora_' in u])}")


## Stage 2 — Micro-SFT on Sonnet 4.6 Thinking Traces (optional)

Skip if `DO_MICRO_SFT_SONNET == 0`. Recommended for a 1-epoch, very-low-LR
pass on 4 k premium thinking traces *before* RL — grafts the reasoning style
without overfitting (LR is tiny, epochs = 1).

Expected schema of `SONNET_TRACES_PATH`:
- `prompt`, `answer`, and one of `generated_cot` / `reasoning` / `cot`.


In [ ]:
if TRAIN_ON_KAGGLE and DO_MICRO_SFT_SONNET:
    import pandas as pd, gc, time, torch, re
    from datasets import Dataset as HFDataset
    from trl import SFTTrainer, SFTConfig

    sdf = pd.read_csv(SONNET_TRACES_PATH)
    cot_col = next(
        (c for c in ("generated_cot", "reasoning", "cot", "thinking") if c in sdf.columns),
        None,
    )
    if cot_col is None:
        raise ValueError(f"Sonnet CSV missing CoT column. Cols: {list(sdf.columns)}")
    sdf = sdf.dropna(subset=["prompt", "answer", cot_col]).reset_index(drop=True)
    print(f"Sonnet traces: {len(sdf)} rows (CoT col={cot_col})")

    PROMPT_SUFFIX = "\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`"
    SYSTEM_PROMPT = (
        "You are an expert mathematics assistant. "
        "Think carefully step by step inside <think>...</think> tags, "
        "then give your final answer inside \\boxed{}."
    )

    records = []
    for _, row in sdf.iterrows():
        cot = str(row[cot_col]).strip()
        if len(cot) < 20:
            continue
        ans = str(row["answer"]).strip()
        # If the trace already contains <think>...</think>, keep it; otherwise wrap.
        if "<think>" not in cot:
            cot = f"<think>\n{cot}\n</think>"
        # Strip any pre-existing \boxed{...} in CoT (we re-attach gold answer).
        cot = re.sub(r"\\boxed\{[^}]*\}", "", cot).rstrip()
        records.append({"messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": str(row["prompt"]) + PROMPT_SUFFIX},
            {"role": "assistant", "content": f"{cot}\n\\boxed{{{ans}}}"},
        ]})
    print(f"Sonnet SFT records: {len(records)}")
    sonnet_ds = HFDataset.from_list(records)

    def fmt(example):
        msgs = example["messages"]
        convs = [msgs] if (msgs and isinstance(msgs[0], dict)) else msgs
        texts = []
        for conv in convs:
            try:
                t = tokenizer.apply_chat_template(
                    conv, tokenize=False, add_generation_prompt=False, enable_thinking=True
                )
            except TypeError:
                t = tokenizer.apply_chat_template(
                    conv, tokenize=False, add_generation_prompt=False
                )
            texts.append(t)
        return texts

    micro_sft_args = SFTConfig(
        output_dir="/kaggle/working/sonnet_sft",
        num_train_epochs=1,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=1e-5,         # tiny — graft, not retrain
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        max_length=8192,
        optim="paged_adamw_8bit",
        weight_decay=0.01,
        max_grad_norm=0.5,
        logging_steps=10,
        save_strategy="no",
        bf16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": True},
        remove_unused_columns=False,
        report_to="none",
        seed=42,
    )

    torch.cuda.empty_cache(); gc.collect()
    print("Starting micro-SFT on Sonnet traces...")
    t0 = time.time()
    SFTTrainer(
        model=model,
        args=micro_sft_args,
        train_dataset=sonnet_ds,
        processing_class=tokenizer,
        formatting_func=fmt,
    ).train()
    print(f"Micro-SFT done in {(time.time()-t0)/60:.1f} min.")
    model.save_pretrained("/kaggle/working/sft_after_sonnet")
    tokenizer.save_pretrained("/kaggle/working/sft_after_sonnet")
else:
    print("Skipping micro-SFT on Sonnet traces.")


## Reward Functions (de-saturated for 85% baseline)

In [ ]:
if TRAIN_ON_KAGGLE:
    import re

    def _extract_boxed(text):
        pos = text.rfind(r"\boxed{")
        if pos == -1:
            return None
        start = pos + len(r"\boxed{")
        depth, i = 1, start
        while i < len(text) and depth > 0:
            if text[i] == "{": depth += 1
            elif text[i] == "}": depth -= 1
            i += 1
        return text[start:i-1].strip() if depth == 0 else None

    def _normalize(ans):
        if not ans: return ""
        ans = ans.strip()
        for tok in (r"\\,", r"\,", r"\\!", r"\!", r"\\ ", r"\ ", r"\text{", "}"):
            ans = ans.replace(tok, "")
        try:
            val = float(ans.replace(",", ""))
            if val == int(val) and abs(val) < 1e15:
                return str(int(val))
            return f"{val:.8g}"
        except (ValueError, OverflowError):
            return ans.lower().replace(" ", "")

    def _is_correct(pred, gold):
        if pred is None: return False
        p, g = _normalize(pred), _normalize(str(gold))
        if p == g: return True
        try:
            pv, gv = float(p.replace(",", "")), float(g.replace(",", ""))
            return abs(pv - gv) < 1e-6 or (gv != 0 and abs(pv - gv) / abs(gv) < 1e-5)
        except (ValueError, OverflowError):
            return False

    # ------------------------------------------------------------------
    # Reward weights tuned for an already-strong (85%) policy.
    # Logic:
    #   * Correctness signal saturates → reduced max to 1.5
    #   * Format already learned → reduced max to 0.3
    #   * Add reasoning-quality signal that discriminates among already-
    #     correct rollouts: think-block existence, structure, no echo,
    #     plausible step count.
    # ------------------------------------------------------------------
    def reward_correctness(completions, answer, **kwargs):
        return [1.5 if _is_correct(_extract_boxed(c), str(a)) else 0.0
                for c, a in zip(completions, answer)]

    def reward_format(completions, **kwargs):
        out = []
        for c in completions:
            s = 0.0
            has_open  = "<think>" in c
            has_close = "</think>" in c
            has_boxed = r"\boxed{" in c
            extracted = _extract_boxed(c)
            if has_open and has_close: s += 0.10
            if has_boxed:              s += 0.10
            if extracted:              s += 0.05
            if has_open and has_close and has_boxed:
                te = c.rfind("</think>"); bp = c.rfind(r"\boxed{")
                if bp > te > 0:         s += 0.05
            out.append(min(s, 0.3))
        return out

    def reward_reasoning_quality(completions, **kwargs):
        """Discriminates among already-correct rollouts. Max 0.5.
        - has a non-trivial <think> block (≥ 200 chars)
        - contains step markers / equations
        - does not just echo the prompt
        """
        out = []
        for c in completions:
            s = 0.0
            m = re.search(r"<think>(.*?)</think>", c, re.DOTALL)
            think = m.group(1) if m else ""
            n = len(think)
            if n >= 200:
                s += 0.15
            if n >= 800:
                s += 0.10
            # Step markers
            steps = len(re.findall(r"(?:Step\s*\d+|^\d+\.|^-)", think, re.MULTILINE))
            if steps >= 3:
                s += 0.10
            # Equation / arithmetic density
            eqs = len(re.findall(r"[=≈]", think))
            if eqs >= 3:
                s += 0.10
            # Penalise pure echo: very high uniq-word ratio is good
            words = think.split()
            if words:
                uniq = len(set(words)) / len(words)
                if uniq > 0.55:
                    s += 0.05
            out.append(min(s, 0.5))
        return out

    def reward_reasoning_length(completions, **kwargs):
        # Soft length bonus (max 0.2). Penalty zone shifted right vs v9-1:
        # we are post-SFT, so we tolerate longer CoT.
        RAMP, PLAT, DECAY = 1500, 6000, 9000
        out = []
        for c in completions:
            m = re.search(r"<think>(.*?)</think>", c, re.DOTALL)
            n = len(m.group(1)) if m else 0
            if n <= RAMP:        s = 0.2 * n / RAMP
            elif n <= PLAT:      s = 0.2
            elif n <= DECAY:     s = 0.2 * (DECAY - n) / (DECAY - PLAT)
            else:                s = 0.0
            out.append(s)
        return out

    def reward_no_repetition(completions, **kwargs):
        out = []
        for c in completions:
            w = c.split()
            if len(w) < 30:
                out.append(0.0); continue
            tail = w[-min(len(w), 200):]
            ngr = [tuple(tail[i:i+6]) for i in range(len(tail)-5)]
            if not ngr:
                out.append(0.0); continue
            div = len(set(ngr)) / len(ngr)
            if   div >= 0.85: out.append(0.10)
            elif div >= 0.65: out.append(0.0)
            elif div >= 0.45: out.append(-0.15)
            else:             out.append(-0.30)
        return out

    def reward_combined(completions, answer, **kwargs):
        cor = reward_correctness(completions, answer, **kwargs)
        fmt = reward_format(completions, **kwargs)
        qua = reward_reasoning_quality(completions, **kwargs)
        lng = reward_reasoning_length(completions, **kwargs)
        rep = reward_no_repetition(completions, **kwargs)
        return [a+b+c+d+e for a,b,c,d,e in zip(cor, fmt, qua, lng, rep)]

    REWARD_FUNCS = [reward_combined]
    print("Reward fn ready: correctness 1.5 + format 0.3 + quality 0.5 + length 0.2 + rep ±")


## Stage 3 — Curriculum Filter (optional)

At 85% accuracy, ≥ 50% of prompts give the model a correct rollout almost
every time → low gradient signal. We pre-score the current (post-Sonnet-SFT)
policy on a sample of GRPO prompts and keep only the ones with **pass@1
in [0.25, 0.85]** — the sweet spot where GRPO learns the most.


In [ ]:
if TRAIN_ON_KAGGLE and DO_CURRICULUM_FILTER:
    import pandas as pd, torch, gc, time, random

    df = pd.read_csv(GRPO_DATASET_PATH)
    df = df.dropna(subset=["answer", "prompt"]).reset_index(drop=True)
    print(f"GRPO source dataset: {len(df)} rows")

    # Score a sample of N_SCORE prompts with K_SCORE rollouts each.
    # 4 k rows is plenty; full pass takes too long, so subsample.
    N_SCORE  = min(2000, len(df))
    K_SCORE  = 3            # rollouts per prompt for pass-rate estimate
    rng      = random.Random(42)
    idxs     = rng.sample(range(len(df)), N_SCORE)
    sample   = df.iloc[idxs].reset_index(drop=True)

    SYSTEM_PROMPT = (
        "You are an expert mathematics assistant. "
        "Think carefully step by step inside <think>...</think> tags, "
        "then give your final answer inside \\boxed{}."
    )
    PROMPT_SUFFIX = "\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`"

    def build_prompt(text):
        msgs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": str(text) + PROMPT_SUFFIX},
        ]
        try:
            return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=True)
        except TypeError:
            return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

    @torch.no_grad()
    def quick_pass_rate(prompt, gold, k):
        text = build_prompt(prompt)
        enc  = tokenizer([text] * k, return_tensors="pt", padding=True, truncation=True, max_length=2048).to(model.device)
        out  = model.generate(
            **enc, max_new_tokens=1024, do_sample=True,
            temperature=0.9, top_p=0.95,
            pad_token_id=tokenizer.pad_token_id, use_cache=True,
        )
        gen = tokenizer.batch_decode(out[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
        hits = sum(_is_correct(_extract_boxed(g), str(gold)) for g in gen)
        return hits / k

    pass_rates = []
    model.eval()
    torch.cuda.empty_cache(); gc.collect()
    t0 = time.time()
    for i, row in sample.iterrows():
        try:
            pr = quick_pass_rate(row["prompt"], row["answer"], K_SCORE)
        except Exception:
            pr = 0.5  # neutral on failure
        pass_rates.append(pr)
        if (i+1) % 50 == 0:
            print(f"  scored {i+1}/{N_SCORE}  ({(time.time()-t0)/60:.1f} min)")
    sample["pass_rate"] = pass_rates

    LO, HI = 0.25, 0.85
    kept = sample[(sample["pass_rate"] >= LO) & (sample["pass_rate"] <= HI)].reset_index(drop=True)
    print(f"Curriculum kept {len(kept)}/{N_SCORE} prompts in [{LO}, {HI}].")
    grpo_df = kept[["prompt", "answer"]].copy()
    grpo_df.to_csv("/kaggle/working/grpo_curriculum.csv", index=False)
else:
    import pandas as pd
    df = pd.read_csv(GRPO_DATASET_PATH).dropna(subset=["answer", "prompt"])
    grpo_df = df[["prompt", "answer"]].sample(frac=1, random_state=42).reset_index(drop=True)
    print(f"Skipping curriculum filter; using full set: {len(grpo_df)} rows")


## Stage 4 — GRPO Training

In [ ]:
if TRAIN_ON_KAGGLE and DO_GRPO:
    import gc, time, subprocess
    import torch
    from datasets import Dataset as HFDataset
    from trl import GRPOTrainer, GRPOConfig
    from transformers import TrainerCallback

    SEED = 42
    PROMPT_SUFFIX = "\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`"
    SYSTEM_PROMPT = (
        "You are an expert mathematics assistant. "
        "Think carefully step by step inside <think>...</think> tags, "
        "then give your final answer inside \\boxed{}."
    )

    grpo_records = [
        {
            "prompt": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": str(p) + PROMPT_SUFFIX},
            ],
            "answer": str(a),
        }
        for p, a in zip(grpo_df["prompt"], grpo_df["answer"])
    ]
    grpo_dataset = HFDataset.from_list(grpo_records)
    print(f"GRPO dataset: {len(grpo_dataset)} prompts")

    # ---- GPU-metrics callback (lightweight) ---------------------------
    class GPUMetricsCallback(TrainerCallback):
        def on_log(self, args, state, control, logs=None, **kwargs):
            if logs is None or not torch.cuda.is_available():
                return
            logs["gpu/mem_alloc_gb"]     = torch.cuda.memory_allocated()     / 2**30
            logs["gpu/mem_reserved_gb"]  = torch.cuda.memory_reserved()      / 2**30
            logs["gpu/mem_peak_gb"]      = torch.cuda.max_memory_allocated() / 2**30

    # ---- GRPO config -------------------------------------------------
    # vLLM colocate keeps both training and rollout on the same GPU.
    # Set use_vllm=False to fall back to HF generate (3-5× slower).
    USE_VLLM = bool(globals().get("VLLM_OK", False))

    grpo_config_kwargs = dict(
        # -- Core GRPO --
        num_generations=8,                # vs v9-1's 4: lower variance
        max_prompt_length=2048,
        max_completion_length=3072,
        temperature=0.9,                  # gentle exploration above SFT mode
        top_p=0.95,
        beta=0.05,                        # stronger KL anchor than v9-1 (0.02)

        # -- DAPO-style asymmetric clip --
        # epsilon_high > epsilon (low) rewards rare correct tokens without
        # blowing up gradient magnitude on common ones.
        epsilon=0.20,
        epsilon_high=0.28,

        # -- Optimization --
        learning_rate=2e-6,               # vs v9-1's 5e-6: 85% baseline → cautious
        lr_scheduler_type="cosine",
        warmup_ratio=0.10,
        num_train_epochs=1,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,    # effective batch = 8 prompts × 8 gen = 64 completions
        optim="paged_adamw_8bit",
        adam_beta1=0.9,
        adam_beta2=0.999,
        adam_epsilon=1e-8,
        weight_decay=0.01,
        max_grad_norm=0.3,

        # -- Memory --
        gradient_checkpointing=True,
        bf16=True,

        # -- Logging --
        output_dir="/kaggle/working/grpo_v11_output",
        logging_steps=1,
        logging_dir="/kaggle/working/tb_logs_v11",
        report_to="tensorboard",
        save_strategy="no",

        # -- Misc --
        seed=SEED,
        remove_unused_columns=False,
        dataloader_num_workers=2,
    )
    if USE_VLLM:
        grpo_config_kwargs.update(
            use_vllm=True,
            vllm_mode="colocate",
            vllm_gpu_memory_utilization=0.35,  # leave room for training
        )

    grpo_config = GRPOConfig(**grpo_config_kwargs)

    print("=" * 60)
    print(f"  GRPO v11 — LR={grpo_config.learning_rate}, beta={grpo_config.beta}")
    print(f"  gens={grpo_config.num_generations}, eps_low/high={grpo_config.epsilon}/{grpo_config.epsilon_high}")
    print(f"  max_completion={grpo_config.max_completion_length}, vLLM={USE_VLLM}")
    print(f"  effective batch = {grpo_config.per_device_train_batch_size}"
          f" × {grpo_config.gradient_accumulation_steps}"
          f" × {grpo_config.num_generations}"
          f" = {grpo_config.per_device_train_batch_size*grpo_config.gradient_accumulation_steps*grpo_config.num_generations} completions/update")
    print("=" * 60)

    torch.cuda.empty_cache(); gc.collect()
    trainer = GRPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=REWARD_FUNCS,
        args=grpo_config,
        train_dataset=grpo_dataset,
        callbacks=[GPUMetricsCallback()],
    )

    print("Starting GRPO v11 training...")
    t0 = time.time()
    trainer.train()
    print(f"GRPO training done in {(time.time()-t0)/60:.1f} min")

    ADAPTER_DIR = "/kaggle/working/sft_adapter"
    model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
    print(f"Adapter saved to {ADAPTER_DIR}")
else:
    print("Skipping GRPO stage.")


## Mode B: Load Pre-trained LoRA

In [ ]:
if USE_PRETRAINED:
    import os
    SRC_ADAPTER_DIR = PRETRAINED_ADAPTER_DATASET_PATH
    required_files = ["adapter_config.json", "adapter_model.safetensors"]
    print("Using pre-trained adapter from:", SRC_ADAPTER_DIR)
    for fname in required_files:
        fpath = os.path.join(SRC_ADAPTER_DIR, fname)
        if not os.path.exists(fpath):
            raise FileNotFoundError(f"Missing: {fpath}")
        print(f"  {fname}: {os.path.getsize(fpath)/1024/1024:.1f} MB")


## Create submission.zip

In [ ]:
import json, os, shutil, zipfile

OUTPUT_DIR = "/kaggle/working"
SUBMISSION_ADAPTER_DIR = os.path.join(OUTPUT_DIR, "submission_adapter")
os.makedirs(SUBMISSION_ADAPTER_DIR, exist_ok=True)
required_files = ["adapter_config.json", "adapter_model.safetensors"]

if TRAIN_ON_KAGGLE:
    src_adapter_dir = "/kaggle/working/sft_adapter"
    print("Packaging GRPO-trained adapter from:", src_adapter_dir)
else:
    src_adapter_dir = PRETRAINED_ADAPTER_DATASET_PATH
    print("Packaging pre-trained adapter directly from:", src_adapter_dir)

for fname in required_files:
    src = os.path.join(src_adapter_dir, fname)
    dst = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
    if not os.path.exists(src):
        raise FileNotFoundError(f"Missing: {src}")
    shutil.copy2(src, dst)
    print(f"Copied {fname} ({os.path.getsize(dst)/1024/1024:.1f} MB)")

config_path = os.path.join(SUBMISSION_ADAPTER_DIR, "adapter_config.json")
with open(config_path) as f:
    cfg = json.load(f)
cfg["base_model_name_or_path"] = BASE_MODEL_NAME
cfg["inference_mode"] = True
cfg["lora_dropout"] = 0.0
with open(config_path, "w") as f:
    json.dump(cfg, f, indent=2)

zip_path = os.path.join(OUTPUT_DIR, "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in required_files:
        zf.write(os.path.join(SUBMISSION_ADAPTER_DIR, fname), fname)
        print(f"  Added {fname}")

print(f"\nsubmission.zip: {os.path.getsize(zip_path)/1024/1024:.1f} MB")
print("Done.")
